In [5]:
import numpy as np
import pandas as pd
import os

In [6]:
csv_files = [f for f in os.listdir() if f.startswith('SV')]
dfs = [pd.read_csv(f) for f in csv_files]
df = pd.concat(dfs, ignore_index=True)

# save
df.to_csv('all_subjects.csv', index=False)

modeling_path = 'modeling/'
if not os.path.exists(modeling_path):
    os.makedirs(modeling_path)
    
del csv_files

In [7]:
# pool data into just columns of true and reported stimulus number

df_subset = df.drop(['loc_1','loc_2','loc_3','loc_4','loc_5', 'direction', 'trial_no'], axis=1)

In [8]:
keys = ['subj_id', 'blindspot', 'n_flash', 'n_beep']


# split df_subset into flash and beep responses
df_flash = df_subset[df_subset['response_type'] == 1].drop(['response_type'], axis=1)
df_beep = df_subset[df_subset['response_type'] == 2].drop(['response_type'], axis=1)

# Stable order within each condition, then add replicate index
df_flash = df_flash.sort_values(keys).copy()
df_beep  = df_beep.sort_values(keys).copy()
df_flash['rep'] = df_flash.groupby(keys).cumcount()
df_beep['rep']  = df_beep.groupby(keys).cumcount()

# join the two DFs
df_joined = pd.merge(
    df_beep,
    df_flash,
    on=keys + ['rep'],
    how='inner',
    suffixes=('_beep','_flash')
)

del keys, df_beep, df_flash, df_subset

In [9]:
ordered_cols = ['n_flash', 'n_beep', 'response_flash', 'response_beep']


In [10]:
# ALL subjects
# (1) All subjects, both locations
df_all = df_joined.copy()
df_all = df_all[ordered_cols]
# remove headers
df_all_values = df_all.to_numpy()
np.savetxt(f'{modeling_path}df_all_group.csv', df_all_values, delimiter=',', fmt='%d')

# (2) All subjects, control vs blind spot
df_control = df_joined[df_joined['blindspot']==0]
df_blindspot = df_joined[df_joined['blindspot']==1]

df_control = df_control[ordered_cols]
df_blindspot = df_blindspot[ordered_cols]

df_control_values = df_control.to_numpy()
np.savetxt(f'{modeling_path}df_control_group.csv', df_control_values, delimiter=',', fmt='%d')

df_blindspot_values = df_blindspot.to_numpy()
np.savetxt(f'{modeling_path}df_blindspot_group.csv', df_blindspot_values, delimiter=',', fmt='%d')

In [11]:
# Per subject
# (3) Per subject, locations combined
unique_subjects = df_joined['subj_id'].unique()
for subj in unique_subjects:
    df_subj = df_joined[df_joined['subj_id'] == subj]
    df_subj = df_subj[ordered_cols]
    df_subj_values = df_subj.to_numpy()
    np.savetxt(f'{modeling_path}df_all_{subj}.csv', df_subj_values, delimiter=',', fmt='%d')

# (4) Per subject, control vs blind spot
for subj in unique_subjects:
    df_subj = df_joined[df_joined['subj_id'] == subj]
    df_control = df_subj[df_subj['blindspot']==0]
    df_blindspot = df_subj[df_subj['blindspot']==1]
    
    df_control = df_control[ordered_cols]
    df_blindspot = df_blindspot[ordered_cols]
    
    df_control_values = df_control.to_numpy()
    np.savetxt(f'{modeling_path}df_control_{subj}.csv', df_control_values, delimiter=',', fmt='%d')
    
    df_blindspot_values = df_blindspot.to_numpy()
    np.savetxt(f'{modeling_path}df_blindspot_{subj}.csv', df_blindspot_values, delimiter=',', fmt='%d')